# 04 Model Training

Purpose: train the first churn prediction model using the feature store dataset.


## Model Training Flow

`Feature Store -> Encoding -> X and y -> Train/Test Split -> Logistic Regression -> Predictions -> Evaluation`


In [5]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


## Load Feature Store


In [6]:
feature_store_df = pd.read_csv("../feature_store/customer_features.csv")

print("Feature Store Shape:", feature_store_df.shape)
feature_store_df.head()


Feature Store Shape: (50000, 16)


,customer_id,age,gender,region,tenure_months,monthly_spend,total_transactions,avg_session_time,support_ticket_count,last_login_days,contract_type,payment_method,churn,engagement_score,spend_per_month,complaint_ratio
0,1,43,Female,South,37,66.04,19,19.98,2,2,Monthly,Credit Card,0,379.62,1.784865,0.052632
1,2,36,Male,North,9,93.26,11,14.68,0,0,Monthly,Credit Card,1,161.48,10.362222,0.000000
2,3,45,Female,East,16,72.42,11,12.48,1,8,One-Year,Credit Card,1,137.28,4.526250,0.058824
3,4,56,Male,West,4,106.27,15,15.50,2,14,Monthly,Bank Transfer,1,232.50,26.567500,0.400000
4,5,35,Female,West,7,66.48,16,13.36,4,3,Monthly,Credit Card,1,213.76,9.497143,0.500000


## One-Hot Encoding

Machine learning models need numerical input.

Categorical columns like gender, region, contract type, and payment method are converted into numeric binary columns.


In [7]:
categorical_cols = [
    "gender",
    "region",
    "contract_type",
    "payment_method"
]

encoded_df = pd.get_dummies(
    feature_store_df,
    columns=categorical_cols,
    drop_first=True
)

print("Before Encoding:", feature_store_df.shape)
print("After Encoding:", encoded_df.shape)

encoded_df.head()


Before Encoding: (50000, 16)
After Encoding: (50000, 21)


,customer_id,age,tenure_months,monthly_spend,total_transactions,avg_session_time,support_ticket_count,last_login_days,churn,engagement_score,spend_per_month,complaint_ratio,gender_Male,region_North,region_South,region_West,contract_type_One-Year,contract_type_Two-Year,payment_method_Credit Card,payment_method_Debit Card,payment_method_UPI
0,1,43,37,66.04,19,19.98,2,2,0,379.62,1.784865,0.052632,False,False,True,False,False,False,True,False,False
1,2,36,9,93.26,11,14.68,0,0,1,161.48,10.362222,0.000000,True,True,False,False,False,False,True,False,False
2,3,45,16,72.42,11,12.48,1,8,1,137.28,4.526250,0.058824,False,False,False,False,True,False,True,False,False
3,4,56,4,106.27,15,15.50,2,14,1,232.50,26.567500,0.400000,True,False,False,True,False,False,False,False,False
4,5,35,7,66.48,16,13.36,4,3,1,213.76,9.497143,0.500000,False,False,False,True,False,False,True,False,False


## Create X and y

`X` means input features.

`y` means target variable.

Here, target variable is `churn`.


In [8]:
X = encoded_df.drop(
    columns=[
        "customer_id",
        "churn"
    ]
)

y = encoded_df["churn"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (50000, 19)
y Shape: (50000,)


## Train Test Split

`test_size=0.2` means 20% test data and 80% training data.

`random_state=42` keeps the split reproducible.


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (40000, 19)
X_test: (10000, 19)
y_train: (40000,)
y_test: (10000,)


## Train Logistic Regression Baseline Model

Logistic Regression is a simple and explainable classification algorithm.

We use it as the first baseline model.


In [10]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000
)

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression model trained successfully")


Logistic Regression model trained successfully


## Make Predictions


In [11]:
y_pred = logistic_model.predict(X_test)

y_pred[:10]


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

## Model Evaluation - Accuracy


In [12]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


Accuracy: 0.5503


## Confusion Matrix


In [13]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm


array([[ 623, 3894],
       [ 603, 4880]])

## Classification Report


In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.51      0.14      0.22      4517
           1       0.56      0.89      0.68      5483

    accuracy                           0.55     10000
   macro avg       0.53      0.51      0.45     10000
weighted avg       0.53      0.55      0.47     10000



## Interview Notes

### Logistic Regression

Logistic Regression is a classification algorithm used to predict binary outcomes such as churn or non-churn.

### Accuracy

Accuracy measures total correct predictions divided by total predictions.

### Confusion Matrix

A confusion matrix shows correct and incorrect predictions for each class.

### Precision

Precision answers: out of all customers predicted as churn, how many actually churned?

### Recall

Recall answers: out of all actual churn customers, how many did the model correctly identify?

### F1 Score

F1 Score balances precision and recall.


# Logistic Regression Model

In [15]:
from sklearn.ensemble import RandomForestClassifier

**n_estimators specifies the number of decision trees in the Random Forest ensemble. Increasing the number of trees generally improves stability and performance but also increases training time.**

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("RandomForestClassifier Trained Successfully")

RandomForestClassifier Trained Successfully


In [17]:
rf_pred = rf_model.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score

In [19]:
rf_accuracy = accuracy_score(
y_test,
rf_pred
)

print("Random Forest Acuracy:", rf_accuracy)

Random Forest Acuracy: 0.5252


In [20]:
from sklearn.metrics import classification_report

In [21]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.47      0.36      0.40      4517
           1       0.56      0.66      0.61      5483

    accuracy                           0.53     10000
   macro avg       0.51      0.51      0.50     10000
weighted avg       0.52      0.53      0.51     10000



# XGBoost Model

In [22]:
import xgboost
print(xgboost.__version__)

2.1.1


In [23]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost model trained successfully")

XGBoost model trained successfully


In [25]:
xgb_pred=xgb_model.predict(X_test)
xgb_pred[:10]

array([0, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [30]:
from sklearn.metrics import classification_report, accuracy_score

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

print("XGBoost Accuracy:", xgb_accuracy)

print(
    classification_report(
        y_test,
        xgb_pred
    )
)

XGBoost Accuracy: 0.5451
              precision    recall  f1-score   support

           0       0.49      0.17      0.25      4517
           1       0.56      0.85      0.67      5483

    accuracy                           0.55     10000
   macro avg       0.52      0.51      0.46     10000
weighted avg       0.53      0.55      0.48     10000



# Random Forest Model Selection

In [33]:
import joblib
joblib.dump(
    rf_model,
    "../models/random_forest.pkl"
)

print("Selected random_forest Model successfully ")

Selected random_forest Model successfully 


# Model Inference

In [35]:
import joblib

loaded_model = joblib.load("../models/random_forest.pkl")

print("Model loaded successfully")

Model loaded successfully


In [36]:
sample_customer = X_test.iloc[[0]]
sample_customer

,age,tenure_months,monthly_spend,total_transactions,avg_session_time,support_ticket_count,last_login_days,engagement_score,spend_per_month,complaint_ratio,gender_Male,region_North,region_South,region_West,contract_type_One-Year,contract_type_Two-Year,payment_method_Credit Card,payment_method_Debit Card,payment_method_UPI
33553,33,6,88.44,22,25.27,2,1,555.94,14.74,0.285714,True,False,False,True,True,False,False,False,False


In [39]:
prediction = loaded_model.predict(sample_customer)
print("Prediction:", prediction)

Prediction: [1]
